<table style="width: 100%; border-collapse: collapse; border: none; background: #f8fafc; border-left: 6px solid #10b981; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #0f172a; font-size: 2.1em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        Árboles y Bosques para Dummies: Midiendo el Desorden en la Habitación (Gini) 🧸
      </h1>
      <p style="margin: 6px 0 0 0; color: #10b981; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Ruta Didáctica: Para Dummies 🧸 | Guía Didáctica
      </p>
      <p style="margin: 4px 0 0 0; color: #64748b; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #059669; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        Módulo 04 • Data Mining
      </span><br>
      <span style="color: #64748b; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #10b981; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Data%20Mining/04%20-%20Arboles%20de%20Decision%20y%20Bosques%20Aleatorios/Para%20Dummies/01_Gini_Dummies.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

## ¿Qué vamos a aprender aquí? 🍬

En el Cuaderno 00 viste que un árbol de decisión "juega a las 20 preguntas" y que el algoritmo CART elige, en cada paso, la "mejor pregunta posible" — pero nunca dijimos exactamente cómo mide CART qué tan buena es una pregunta. Ese es el tema de este cuaderno, y la clave para entenderlo es una idea muy simple: **medir qué tan mezclado está un grupo**.

Al terminar este cuaderno vas a poder responder:
1. ¿Qué significa que un grupo de datos esté "puro" o "mezclado", y cómo se mide con un número?
2. ¿Por qué una buena pregunta reduce esa mezcla?
3. ¿Qué otras "recetas" existen para medir lo mismo, y cuándo se usa cada una?

---
## 1. La Gran Analogía: La Bolsa de Dulces Mezclados 🍬

Imagina tu habitación desordenada, y en el piso hay bolsitas de dulces que se mezclaron sin querer. Ahora piensa en tres bolsas distintas:

* **Bolsa A:** 10 dulces de cereza, 0 de limón. Si metes la mano y sacas uno al azar, sabes con total seguridad qué sabor te va a tocar: cereza. Esta bolsa está **pura** — no hay mezcla, no hay sorpresa.
* **Bolsa B:** 8 dulces de cereza, 2 de limón. Casi siempre sacarías cereza, pero ya hay algo de incertidumbre. Un poco mezclada.
* **Bolsa C:** 5 dulces de cereza, 5 de limón. Aquí es un 50/50 total: no tienes ni idea de qué sabor te va a tocar. Esta es la bolsa **más mezclada posible** entre dos sabores.

La **impureza de Gini** es, ni más ni menos, una fórmula que le pone un número exacto a esta idea de "qué tan mezclada está la bolsa": vale **0** cuando la bolsa es pura (como la Bolsa A) y llega a su **máximo** cuando la mezcla es 50/50 (como la Bolsa C). Cuando un árbol de decisión evalúa una pregunta candidata, en el fondo está preguntando: *"¿esta pregunta separa los dulces por sabor, dejando bolsas más puras que antes?"*

In [ ]:
def gini(proporciones):
    """Impureza de Gini: 1 menos la suma de cada proporcion al cuadrado."""
    return 1 - sum(p ** 2 for p in proporciones)


bolsa_pura = [1.0, 0.0]        # 10 cereza, 0 limon
bolsa_poco_mixta = [0.8, 0.2]  # 8 cereza, 2 limon
bolsa_mixta = [0.5, 0.5]       # 5 cereza, 5 limon

print(f"Bolsa A (10 cereza, 0 limon)  -> Gini = {gini(bolsa_pura):.2f}")
print(f"Bolsa B (8 cereza, 2 limon)   -> Gini = {gini(bolsa_poco_mixta):.2f}")
print(f"Bolsa C (5 cereza, 5 limon)   -> Gini = {gini(bolsa_mixta):.2f}")

### 🤔 ¿Qué acaba de pasar?
Los números confirman exactamente la intuición: la Bolsa A (pura) tiene Gini = 0.00, la Bolsa C (la más mezclada posible entre dos sabores) tiene Gini = 0.50 — el máximo en este caso de dos categorías — y la Bolsa B, a medio camino, queda en 0.32. La fórmula que acabamos de programar es la fórmula real que usa CART:

$$\text{Gini} = 1 - \sum_{k} p_k^2$$

donde cada $p_k$ es la proporción de cada sabor (o clase) en la bolsa (o nodo). No hace falta memorizar la fórmula para entender la idea: es simplemente "qué tan fácil es adivinar al azar qué hay en la bolsa" — fácil si está pura, difícil si está mezclada.

---
## 2. Ahora con datos reales: la bolsa de clientes que se van 📡

Cambiemos de dulces a clientes. Piensa en los 2500 clientes de telecomunicaciones (`abandono_clientes_telecom.csv`) como una gran bolsa mezclada de dos "sabores": los que se quedan y los que se van (`churn`). ¿Qué tan mezclada está esa bolsa?

In [ ]:
import os, urllib.request, urllib.parse, gzip, base64

_DATA_EMBEDDED = {
    "abandono_clientes_telecom.csv": "...",
}

def load_dataset(filename: str) -> str:
    candidates = [
        os.path.join("data", filename),
        os.path.join("..", "data", filename),
        os.path.join("04 - Arboles de Decision y Bosques Aleatorios", "data", filename),
        os.path.join("Data Mining", "04 - Arboles de Decision y Bosques Aleatorios", "data", filename),
        os.path.join("..", "04 - Arboles de Decision y Bosques Aleatorios", "data", filename),
        filename,
    ]
    for p in candidates:
        if os.path.exists(p):
            return p
    os.makedirs("data", exist_ok=True)
    target = os.path.join("data", filename)
    base_url = "https://raw.githubusercontent.com/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/main/Data%20Mining/04%20-%20Arboles%20de%20Decision%20y%20Bosques%20Aleatorios/data/"
    url = base_url + urllib.parse.quote(filename)
    try:
        urllib.request.urlretrieve(url, target, timeout=3)
        return target
    except Exception:
        pass
    if filename in _DATA_EMBEDDED:
        raw = gzip.decompress(base64.b64decode(_DATA_EMBEDDED[filename]))
        with open(target, "wb") as f:
            f.write(raw)
        return target
    raise FileNotFoundError(f"No se pudo obtener el dataset: {filename}")

import pandas as pd, numpy as np, os
from sklearn.tree import DecisionTreeClassifier

clientes = pd.read_csv(load_dataset("abandono_clientes_telecom.csv"))

p_se_queda = (clientes["churn"] == 0).mean()
p_se_va = (clientes["churn"] == 1).mean()
gini_general = gini([p_se_queda, p_se_va])

print(f"Bolsa general de {len(clientes)} clientes:")
print(f"  {p_se_queda:.2%} se quedan (sabor 'cereza')")
print(f"  {p_se_va:.2%} se van (sabor 'limon')")
print(f"  Gini de la bolsa general = {gini_general:.4f}")

### 🤔 ¿Qué acaba de pasar?
La bolsa completa de 2500 clientes tiene un Gini de **0.4108** — bastante cerca del máximo posible de 0.5 (que sería un 50/50 exacto). Tiene sentido: con 71.12% que se quedan y 28.88% que se van, la bolsa está lejos de ser pura, aunque tampoco es un empate perfecto. Este 0.4108 es el número de referencia contra el que CART va a comparar cada pregunta candidata: si una pregunta logra bajar ese número, es una pregunta útil.

---
## 3. ¿Podemos hacer menos mezclada la bolsa? 🔀

Probemos la primera pregunta real que eligió el árbol del Cuaderno 00: *"¿el cliente tiene contrato Mes a Mes?"*. Esa pregunta separa la bolsa grande de 2500 clientes en dos bolsas más pequeñas. Si Gini realmente mide "mezcla", cada una de esas dos bolsas nuevas debería quedar menos mezclada que la bolsa original.

In [ ]:
es_mes_a_mes = clientes["tipo_contrato"] == "Mes_a_Mes"
grupo_mes_a_mes = clientes[es_mes_a_mes]
grupo_resto = clientes[~es_mes_a_mes]

p_va_mes = (grupo_mes_a_mes["churn"] == 1).mean()
p_va_resto = (grupo_resto["churn"] == 1).mean()
gini_mes = gini([1 - p_va_mes, p_va_mes])
gini_resto = gini([1 - p_va_resto, p_va_resto])

n = len(clientes)
n_mes, n_resto = len(grupo_mes_a_mes), len(grupo_resto)
gini_ponderado = (n_mes / n) * gini_mes + (n_resto / n) * gini_resto

print(f"Bolsa 'Mes a Mes':        n={n_mes}   se va={p_va_mes:.2%}   Gini={gini_mes:.4f}")
print(f"Bolsa 'Un_Ano/Dos_Anos':  n={n_resto}   se va={p_va_resto:.2%}   Gini={gini_resto:.4f}")
print(f"\nGini ponderado de las dos bolsas nuevas = {gini_ponderado:.4f}")
print(f"Gini de la bolsa original                = {gini_general:.4f}")
print(f"Reduccion de mezcla lograda por la pregunta = {gini_general - gini_ponderado:.4f}")

### 🤔 ¿Qué acaba de pasar?
La pregunta sí funciona: la bolsa "Mes a Mes" queda con Gini = 0.4874 (todavía bastante mezclada — 42.06% se va) pero la bolsa "Un_Ano/Dos_Anos" queda mucho más pura, con Gini = 0.2213 (solo 12.67% se va). Promediando las dos bolsas nuevas según su tamaño, el Gini baja de 0.4108 a 0.3681 — una reducción de 0.0427. Esa reducción es exactamente lo que CART calcula, para cada pregunta candidata sobre cada variable, antes de quedarse con la que más reduce la mezcla. No es casualidad que esta haya sido la primera pregunta que el árbol del Cuaderno 00 eligió hacer: de todas las preguntas posibles, fue la que más redujo el Gini.

---
## 4. Otra receta parecida: la Entropía 🔬

Gini no es la única forma de medir "qué tan mezclada" está una bolsa. Existe otra fórmula, llamada **Entropía**, que viene de la teoría de la información y mide algo muy similar: qué tanta "sorpresa" hay al sacar un dulce al azar de la bolsa. Al igual que Gini, la Entropía vale 0 cuando la bolsa es pura y llega a su máximo cuando la mezcla es 50/50 — solo que su escala máxima es distinta (1.0 en vez de 0.5 para dos categorías).

No hace falta memorizar su fórmula para seguir este módulo; basta con reconocerla si la ves en otro lado y saber que, en la práctica, **casi siempre está de acuerdo con Gini** sobre cuál es la mejor pregunta. Verifiquemos ambas cosas con la bolsa general de clientes.

In [ ]:
def entropia(proporciones):
    """Entropia de Shannon: mide 'sorpresa promedio', igual que Gini mide mezcla."""
    return -sum(p * np.log2(p) for p in proporciones if p > 0)


entropia_general = entropia([p_se_queda, p_se_va])
print(f"Entropia de la bolsa general = {entropia_general:.4f}  (Gini de la misma bolsa = {gini_general:.4f})")

# Verificacion: comparamos nuestro Gini "a mano" contra el que calcula scikit-learn internamente
datos_verif = clientes.copy()
datos_verif["contrato_mes_a_mes"] = (datos_verif["tipo_contrato"] == "Mes_a_Mes").astype(int)
arbol_verif = DecisionTreeClassifier(max_depth=1, random_state=42)
arbol_verif.fit(datos_verif[["contrato_mes_a_mes"]], datos_verif["churn"])

print(f"\nGini calculado a mano para la raiz:        {gini_general:.4f}")
print(f"Gini que reporta scikit-learn para la raiz: {arbol_verif.tree_.impurity[0]:.4f}")

### 🤔 ¿Qué acaba de pasar?
La Entropía de la misma bolsa general da 0.8672 — un número distinto al Gini (0.4108) porque usa una escala distinta, pero cuenta la misma historia: la bolsa está bastante mezclada, no muy lejos de su máximo teórico. Y la verificación final confirma algo importante: el Gini que calculamos "a mano" (0.4108) coincide exactamente con el que `scikit-learn` calcula internamente al entrenar un árbol de verdad — la fórmula que programamos arriba es, literalmente, la misma que usa CART por dentro.

---
## 5. Un caso especial: cuando se predice un número en vez de una categoría 📈

Gini y Entropía sirven para cuando la respuesta es una categoría ("se va" / "se queda"). Pero si en vez de eso quisieras predecir cuánto va a gastar un cliente en total (`cargos_totales`, un número), ya no tiene sentido hablar de "bolsas de sabores" — en su lugar, un grupo se considera "puro" cuando los números que contiene son parecidos entre sí (poca variación), y se usa una receta distinta llamada **MSE** (error cuadrático medio) para medir esa variación. La idea de fondo es la misma —CART sigue buscando la pregunta que más "ordena" cada grupo resultante— solo que ahora "ordenado" significa "números parecidos" en lugar de "un solo sabor".

---
##### 🛠️ Práctica 1: Prueba otra pregunta: ¿lleva un año o menos?

En el cuaderno probaste la pregunta *"¿tiene contrato Mes a Mes?"* y viste que bajaba la mezcla de la bolsa en 0.0427. Ahora tú pruebas **otra** pregunta candidata sobre la misma bolsa `clientes`: *"¿el cliente lleva 12 meses o menos (`antiguedad_meses <= 12`)?"*.

1. Separa la bolsa en dos: `grupo_nuevo` (12 meses o menos) y `grupo_antiguo` (el resto).
2. Calcula el porcentaje que se va en cada grupo y, con tu función `gini`, la impureza de cada uno (`gini_nuevo`, `gini_antiguo`).
3. Calcula el Gini ponderado de las dos bolsas (cada una pesa según su tamaño) y guárdalo en `gini_pond_antig`. La reducción de mezcla es `reduccion_antig = gini_general - gini_pond_antig`.
4. Compárala con la reducción de la pregunta del contrato (`reduccion_contrato`, ya te la dejamos calculada). ¿Cuál de las dos preguntas ordena mejor la bolsa?

In [ ]:
# Escribe tu código aquí

# La reducción de la pregunta del contrato, que ya calculaste arriba
reduccion_contrato = gini_general - gini_ponderado

# 1) Separar la bolsa con la nueva pregunta
# es_nuevo = clientes["antiguedad_meses"] <= ...
# grupo_nuevo = clientes[es_nuevo]
# grupo_antiguo = clientes[~es_nuevo]

# 2) Qué porcentaje se va en cada grupo y qué tan mezclado queda
# p_va_nuevo = (grupo_nuevo["churn"] == 1).mean()
# p_va_antiguo = (grupo_antiguo["churn"] == 1).mean()
# gini_nuevo = gini([1 - p_va_nuevo, p_va_nuevo])
# gini_antiguo = gini([1 - p_va_antiguo, ...])

# 3) Gini ponderado (cada bolsa pesa según su tamaño) y reducción
# gini_pond_antig = (len(grupo_nuevo) / len(clientes)) * gini_nuevo + (len(grupo_antiguo) / len(clientes)) * ...
# reduccion_antig = gini_general - gini_pond_antig

# 4) Comparar las dos preguntas
# print(f"Reducción con 'antigüedad <= 12': {reduccion_antig:.4f}")
# print(f"Reducción con 'contrato Mes a Mes': {reduccion_contrato:.4f}")


<details>
<summary><b>💡 Haz clic aquí para ver la solución guiada...</b></summary>

```python
# La reducción de la pregunta del contrato, que ya calculaste arriba
reduccion_contrato = gini_general - gini_ponderado

# 1) Separar la bolsa con la nueva pregunta
es_nuevo = clientes["antiguedad_meses"] <= 12
grupo_nuevo = clientes[es_nuevo]
grupo_antiguo = clientes[~es_nuevo]

# 2) Qué porcentaje se va en cada grupo y qué tan mezclado queda
p_va_nuevo = (grupo_nuevo["churn"] == 1).mean()
p_va_antiguo = (grupo_antiguo["churn"] == 1).mean()
gini_nuevo = gini([1 - p_va_nuevo, p_va_nuevo])
gini_antiguo = gini([1 - p_va_antiguo, p_va_antiguo])
print(f"12 meses o menos: n={len(grupo_nuevo)}  se va={p_va_nuevo:.2%}  Gini={gini_nuevo:.4f}")
print(f"Más de 12 meses:  n={len(grupo_antiguo)}  se va={p_va_antiguo:.2%}  Gini={gini_antiguo:.4f}")

# 3) Gini ponderado (cada bolsa pesa según su tamaño) y reducción
gini_pond_antig = (len(grupo_nuevo) / len(clientes)) * gini_nuevo + (len(grupo_antiguo) / len(clientes)) * gini_antiguo
reduccion_antig = gini_general - gini_pond_antig

# 4) Comparar las dos preguntas
print(f"\nReducción con 'antigüedad <= 12': {reduccion_antig:.4f}")
print(f"Reducción con 'contrato Mes a Mes': {reduccion_contrato:.4f}")
mejor = "contrato Mes a Mes" if reduccion_contrato > reduccion_antig else "antigüedad <= 12"
print(f"La pregunta que mejor ordena la bolsa es: {mejor}")

# Comprobaciones
assert len(grupo_nuevo) + len(grupo_antiguo) == len(clientes)
assert 0 <= gini_pond_antig <= gini_general          # preguntar nunca deja la bolsa más mezclada en promedio
assert reduccion_antig >= 0
```
</details>

---


---
## Resumen Relámpago ⚡

| Idea Clave | Explicación en 5 segundos |
|---|---|
| **Impureza de Gini** | Mide qué tan mezclada está una bolsa (o un grupo de clientes): 0 = pura, máximo = mezcla 50/50. |
| **Fórmula** | $\text{Gini} = 1 - \sum_k p_k^2$, sumando el cuadrado de la proporción de cada categoría. |
| **Cómo elige CART una pregunta** | Prueba muchas preguntas candidatas y se queda con la que más reduce el Gini promedio de los grupos resultantes. |
| **Entropía** | Otra receta para medir lo mismo (mezcla/pureza); en la práctica casi siempre coincide con Gini. |
| **MSE** | La versión de "pureza" para cuando se predice un número en vez de una categoría. |

**Siguiente paso:** ya sabes cómo el árbol construye sus preguntas (Cuaderno 00) y cómo elige la mejor de todas (este cuaderno). El próximo cuaderno de la ruta, **"Cortando Ramas Secas (Poda)"**, te muestra cómo evitar que el árbol se pase de preguntón y termine memorizando en vez de aprender.

---
<div align="center">
  <p style="font-size: 0.9em; color: #64748b;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Ruta Para Dummies: Árboles y Bosques</i>
  </p>
</div>